In [ ]:
import os
import sys

os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F

jdbc_jar_path = "/Users/jatin/Library/DBeaverData/drivers/maven/maven-central/org.postgresql/postgresql-42.7.2.jar"

spark = (
    SparkSession.builder
    .appName("intelligent-ingestion")
    .master("local[*]")
    .config("spark.jars", jdbc_jar_path)
    .getOrCreate()
)

pg_url = "jdbc:postgresql://localhost:5432/Project-1"
pg_properties = {
    "user": "postgres",
    "password": "pass",
    "driver": "org.postgresql.Driver",
}
metadata_table = '"CDAC".file_metadata'

In [ ]:
metadata_schema = (
    "file_id STRING, file_name STRING, file_path STRING, file_size LONG, "
    "file_extension STRING, document_type STRING, status STRING, "
    "ingestion_timestamp TIMESTAMP"
)

try:
    metadata_df = spark.read.jdbc(url=pg_url, table=metadata_table, properties=pg_properties)
except Exception:
    # Table doesn't exist yet (first run) — start from an empty frame.
    metadata_df = spark.createDataFrame([], schema=metadata_schema)

metadata_df.toPandas()

In [ ]:
new_files_df = metadata_df.filter(F.col("status") == "NEW")

new_files_df.toPandas()

In [ ]:
input_path = "../raw"

In [ ]:
from pathlib import Path

raw_dir = Path(input_path)
local_files = [p for p in raw_dir.iterdir() if p.is_file()]

files_df = (
    spark.createDataFrame(
        [(f.name, str(f.resolve()), f.stat().st_size) for f in local_files],
        ["file_name", "file_path", "file_size"]
    )
    .withColumn(
        "file_extension",
        F.lower(F.regexp_extract("file_name", r"\.([^.]+)$", 1))
    )
    .withColumn("document_type", F.lit("UNKNOWN"))
    .withColumn("status", F.lit("NEW"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn(
        "file_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("file_path"),
                F.col("file_size").cast("string")
            ),
            256
        )
    )
    .select(
        "file_id",
        "file_name",
        "file_path",
        "file_size",
        "file_extension",
        "document_type",
        "status",
        "ingestion_timestamp"
    )
)

files_df.toPandas()

In [ ]:
existing_ids = {row.file_id for row in metadata_df.select("file_id").collect()}

# Only append files we haven't seen before — an overwrite here would reset
# the status of files already processed by later pipeline stages.
new_only_df = files_df.filter(~F.col("file_id").isin(existing_ids)) if existing_ids else files_df

new_only_df.write.jdbc(
    url=pg_url,
    table=metadata_table,
    mode="append",
    properties=pg_properties
)

In [ ]:
spark.read.jdbc(url=pg_url, table=metadata_table, properties=pg_properties).toPandas()